In [1]:
from langchain_openai import ChatOpenAI 
import os
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model ="gpt-5-mini") 


### Tools

In [2]:
from langchain.tools import tool

In [3]:
@tool # Decorator to mark this function as a tool
def tool_duckduckgo_search(query: str) -> str:
    """Use this tool when you need to answer questions about current events or general knowledge questions."""
   
    from langchain_community.tools import DuckDuckGoSearchRun

    search = DuckDuckGoSearchRun()

    response = search.invoke(query)
    return response

In [4]:
# tool_duckduckgo_search.invoke("What is the capital of India?")

In [5]:
@tool # Decorator to mark this function as a tool
def tool_wikipedia_search(query: str) -> str:
    """Use this tool when you need to answer questions about persons, places etc."""
    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper
    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    response = wikipedia.run(query)
    return response

In [6]:
# tool_wikipedia_search.invoke("Who is the president of USA?")

In [7]:
@tool
def tool_arxiv_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about scientific papers or research topics. """

    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    # 1. Initialize the arXiv API wrapper
    arxiv_wrapper = ArxivAPIWrapper(
        top_k_results=3,       # Number of papers to retrieve
        doc_content_chars_max=2000  # Max characters per document
    )

    # 2. Create the arXiv tool
    arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

    # 3. Use the tool directly
    result = arxiv_tool.run(query)

    print(result)

In [8]:
# tool_arxiv_search.invoke("What are the latest research papers on deep learning?")

In [9]:
@tool
def tool_personal_info(query: str) -> str:
    """Use this tool when you need to answer questions about personal information. Like name, age, occupation etc.
        Args:
        name (str): The name of the person to look up.
    Returns:
        str: A string containing the person's age and occupation, or a message if the information is not found.
    """
    infos = [{
        "name": "John Doe",
        "age": 30,
        "Occupation": "Software Engineer"
    },
    {
        "name": "Jane Smith",
        "age": 25,
        "Occupation": "Data Scientist"
    }]
    for info in infos:
        if query.lower() in info["name"].lower():
            return f"{info['name']} is a {info['age']} year old and works as a {info['Occupation']}."
    return "No information found."

In [10]:
# tool_personal_info.invoke("John Doe")

In [11]:
if os.path.exists("./chroma_db_semantic"):
    print("Chroma database already exists. Skipping creation.")

Chroma database already exists. Skipping creation.


In [12]:
@tool
def tool_rag_search(query: str) -> str:
    """Use this tool when you need to answer questions based on NovaSphere Organization's documentation. """
    from langchain_community.vectorstores import Chroma
    from langchain_openai import OpenAIEmbeddings

    embed_model = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)
    chroma_db_con = Chroma(persist_directory = "./chroma_db_semantic", embedding_function=embed_model)

    #1. Retrieve relevant documents from the Chroma database
    relevant_docs = chroma_db_con.similarity_search(query, k=2)
    relevant_docs_content = "\n".join([doc.page_content for doc in relevant_docs])

    return relevant_docs_content


In [13]:
# tool_rag_search.invoke("What is NovaSphere Organization?")

### Binding Tools

In [14]:
toolkit = [
    tool_duckduckgo_search,
    tool_wikipedia_search,
    tool_arxiv_search,
    tool_personal_info,
    tool_rag_search
    
]

In [15]:
llm_bind = llm.bind_tools(toolkit) #.bind_tools() is function that binds the tools to the llm. It takes a list of tools as input and returns a new llm that has access to those tools.

In [16]:
# llm_bind.invoke("What is the age of John Doe? Make tool calls if necessary to answer the question.")

### ReAct Agent

In [17]:
from langchain.agents import create_agent

my_agent = create_agent(llm_bind, toolkit)

In [18]:
# my_agent.invoke(
#     {"messages": [{"role": "user", "content": "What is the age of John Doe? Make tool calls if necessary to answer the question."}]}
# )

In [19]:
# my_agent.invoke(
#     {"messages": [{"role": "user", "content": "When was NovaSphere Organization founded? "}]}
# )

### Memorary

In [20]:
agent_memory = []

In [21]:
def invoke_agent_with_memory(query: str):
    response = my_agent.invoke(
        {"messages": [{"role": "user", "content": f"Answer this query:{query}. You can refer to the previous interactions: {str(agent_memory)}"}]}
    )
    agent_memory.extend(response["messages"])
    return response

In [22]:
# invoke_agent_with_memory("When was NovaSphere Organization founded? ")

In [23]:
# invoke_agent_with_memory("Could you please summarize the previous interactions?")

In [24]:
agent_memory

[]

In [25]:
while True:
    query = input("Ask me anything: ")
    response = invoke_agent_with_memory(query)
    print(response)
    flag = input("Do you want to continue? (yes/no): ")
    if flag.lower() == "no":
        break

{'messages': [HumanMessage(content='Answer this query:Hi, My name is Anubhav . You can refer to the previous interactions: []', additional_kwargs={}, response_metadata={}, id='0baca018-c002-4f48-96a0-22319ad60d29'), AIMessage(content='Hi Anubhav — nice to meet you. I don’t see any previous interactions here (the list is empty). How can I help you today?\n\nThings I can do if you want ideas:\n- Answer questions or explain topics\n- Draft or edit emails, messages, or documents\n- Help with coding, debugging, or code review\n- Create plans, checklists, or study notes\n- Brainstorm ideas or give feedback\n\nTell me what you need and any relevant details; I’ll get started.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 240, 'prompt_tokens': 332, 'total_tokens': 572, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens

C:\Users\hp\AppData\Local\Temp\ipykernel_15732\3744645833.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory = "./chroma_db_semantic", embedding_function=embed_model)


{'messages': [HumanMessage(content="Answer this query:What is NovaSphere ?. You can refer to the previous interactions: [HumanMessage(content='Answer this query:Hi, My name is Anubhav . You can refer to the previous interactions: []', additional_kwargs={}, response_metadata={}, id='0baca018-c002-4f48-96a0-22319ad60d29'), AIMessage(content='Hi Anubhav — nice to meet you. I don’t see any previous interactions here (the list is empty). How can I help you today?\\n\\nThings I can do if you want ideas:\\n- Answer questions or explain topics\\n- Draft or edit emails, messages, or documents\\n- Help with coding, debugging, or code review\\n- Create plans, checklists, or study notes\\n- Brainstorm ideas or give feedback\\n\\nTell me what you need and any relevant details; I’ll get started.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 240, 'prompt_tokens': 332, 'total_tokens': 572, 'completion_tokens_details': {'accepted_prediction_tokens': 0, '

In [26]:
agent_memory

[HumanMessage(content='Answer this query:Hi, My name is Anubhav . You can refer to the previous interactions: []', additional_kwargs={}, response_metadata={}, id='0baca018-c002-4f48-96a0-22319ad60d29'),
 AIMessage(content='Hi Anubhav — nice to meet you. I don’t see any previous interactions here (the list is empty). How can I help you today?\n\nThings I can do if you want ideas:\n- Answer questions or explain topics\n- Draft or edit emails, messages, or documents\n- Help with coding, debugging, or code review\n- Create plans, checklists, or study notes\n- Brainstorm ideas or give feedback\n\nTell me what you need and any relevant details; I’ll get started.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 240, 'prompt_tokens': 332, 'total_tokens': 572, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache